# Paper Figures From Local Outputs

This notebook reads the current repo outputs only:

- `outputs/pre_detection/<dataset>/results.csv`
- `outputs/post_detection/<dataset>/results.csv`
- `outputs/post_detection/<dataset>/oof_predictions.csv`
- `outputs/post_detection/<dataset>/post_detection_features.csv`

and regenerates paper-style figures with the updated results format.

Expected post-detection views:

- `conf_only`
- `conf_mt`
- `conf_tta`
- `full`
- `post_plus_fdino_roi`

If your `glass` post-detection outputs still contain older names such as `post_core` or `post_plus_fdino_full`, rerun the post stage first with `--reuse-existing` so the CSV schema matches the current pipeline.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

PRE_ROOT = ROOT / 'outputs' / 'pre_detection'
POST_ROOT = ROOT / 'outputs' / 'post_detection'
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ('ca', 'glass')
DATASET_LABELS = {'ca': 'CA', 'glass': 'Glass'}
DATASET_COLORS = {'ca': '#2563eb', 'glass': '#16a34a'}


In [ ]:
def load_results(stage_root: Path, dataset: str) -> pd.DataFrame:
    path = stage_root / dataset / 'results.csv'
    df = pd.read_csv(path)
    return df.sort_values(['view', 'model']).reset_index(drop=True)


def load_post_predictions(dataset: str) -> pd.DataFrame:
    path = POST_ROOT / dataset / 'oof_predictions.csv'
    return pd.read_csv(path).sort_values('row_id').reset_index(drop=True)


def load_post_features(dataset: str) -> pd.DataFrame:
    path = POST_ROOT / dataset / 'post_detection_features.csv'
    return pd.read_csv(path).sort_values('row_id').reset_index(drop=True)


def require_row(df: pd.DataFrame, view: str, model: str) -> pd.Series:
    rows = df[(df['view'] == view) & (df['model'] == model)]
    if rows.empty:
        raise ValueError(f'Missing row for view={view!r}, model={model!r}')
    return rows.iloc[0]


pre_results = {dataset: load_results(PRE_ROOT, dataset) for dataset in DATASETS}
post_results = {dataset: load_results(POST_ROOT, dataset) for dataset in DATASETS}
post_preds = {dataset: load_post_predictions(dataset) for dataset in DATASETS}
post_feats = {dataset: load_post_features(dataset) for dataset in DATASETS}

for dataset in DATASETS:
    print(f'Loaded {dataset}:')
    print('  pre rows :', len(pre_results[dataset]))
    print('  post rows:', len(post_results[dataset]))
    print('  preds cols:', len(post_preds[dataset].columns))
    print('  feat cols :', len(post_feats[dataset].columns))


## Figure 1

Paper-style pre-vs-post comparison using the current repo outputs.

Selection policy:

- Pre-detection DINO views: `pca256_ridge`
- Post-detection full view: `random_forest`
- Combined post + DINO ROI view: `random_forest`

This mirrors the paper intent, while using only the views that the current repo actually produces.


In [ ]:
comparison_specs = [
    ('DINOv2 ROI', 'pre', 'fdino_roi', 'pca256_ridge'),
    ('DINOv2 full image', 'pre', 'fdino_full', 'pca256_ridge'),
    ('Post-detection full', 'post', 'full', 'random_forest'),
    ('DINOv2 ROI + post', 'post', 'post_plus_fdino_roi', 'random_forest'),
]

rows = []
for dataset in DATASETS:
    for label, stage_key, view, model in comparison_specs:
        source = pre_results[dataset] if stage_key == 'pre' else post_results[dataset]
        row = require_row(source, view, model)
        rows.append({
            'dataset': dataset,
            'dataset_label': DATASET_LABELS[dataset],
            'feature_label': label,
            'stage': row['stage'],
            'view': row['view'],
            'model': row['model'],
            'r2': row['r2'],
            'spearman': row['spearman'],
            'n_features': row['n_features'],
        })

comparison_df = pd.DataFrame(rows)
comparison_df


In [ ]:
feature_order = [spec[0] for spec in comparison_specs]
metrics = [('r2', 'R²'), ('spearman', 'Spearman ρ')]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
bar_width = 0.36
x = np.arange(len(feature_order))

for ax, (metric, title) in zip(axes, metrics):
    for offset, dataset in zip((-bar_width / 2, bar_width / 2), DATASETS):
        subset = comparison_df[comparison_df['dataset'] == dataset].set_index('feature_label').loc[feature_order]
        ax.bar(
            x + offset,
            subset[metric].to_numpy(),
            width=bar_width,
            color=DATASET_COLORS[dataset],
            alpha=0.9,
            label=DATASET_LABELS[dataset],
        )

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(feature_order, rotation=18, ha='right')
    ax.set_ylim(0.0, max(0.7, comparison_df[metric].max() * 1.15))
    ax.grid(axis='y', alpha=0.25)

axes[0].legend(frameon=True, loc='upper left')
fig.suptitle('Pre- vs. post-detection feature comparison', y=1.02, fontsize=14)
fig.tight_layout()

prepost_path = FIG_DIR / 'fig_pre_post_comparison_local.png'
fig.savefig(prepost_path, dpi=160, bbox_inches='tight')
plt.show()
print(prepost_path)


## Figure 2

Risk-coverage curves in the paper style, using only the local post-detection outputs.

Signals:

- detection confidence: `conf_score_mean`
- TTA consistency: `tta_iou_mean`
- meta-regressor: `full__random_forest`

The random band follows Cesar's paper script: 500 shuffled orderings and the 5th-95th percentile envelope.


In [ ]:
def rc_curve(ious: np.ndarray, signal: np.ndarray, coverages: np.ndarray) -> np.ndarray:
    order = np.argsort(signal)[::-1]
    n = len(ious)
    return np.array([
        ious[order[:max(1, int(np.ceil(c * n)))]] .mean()
        for c in coverages
    ])


def random_band(ious: np.ndarray, coverages: np.ndarray, n_shuffles: int = 500, seed: int = SEED):
    n = len(ious)
    rng = np.random.default_rng(seed)
    curves = np.array([
        [ious[rng.permutation(n)[:max(1, int(np.ceil(c * n)))]] .mean() for c in coverages]
        for _ in range(n_shuffles)
    ])
    return np.percentile(curves, [5, 95], axis=0)


def risk_coverage_table(dataset: str, coverages=(0.2, 0.4, 0.6, 0.8, 1.0)) -> pd.DataFrame:
    preds = post_preds[dataset]
    feats = post_feats[dataset]
    merged = preds[['row_id', 'iou_target', 'full__random_forest']].merge(
        feats[['row_id', 'conf_score_mean', 'tta_iou_mean']],
        on='row_id',
        how='inner',
    )
    y = merged['iou_target'].to_numpy(dtype=float)
    conf = merged['conf_score_mean'].to_numpy(dtype=float)
    tta = merged['tta_iou_mean'].to_numpy(dtype=float)
    meta = merged['full__random_forest'].to_numpy(dtype=float)
    baseline = y.mean()
    rows = []
    for c in coverages:
        rows.append({
            'dataset': DATASET_LABELS[dataset],
            'coverage': c,
            'confidence': rc_curve(y, conf, np.array([c]))[0],
            'tta': rc_curve(y, tta, np.array([c]))[0],
            'meta_regressor_rf': rc_curve(y, meta, np.array([c]))[0],
            'all_images_mean': baseline,
        })
    table = pd.DataFrame(rows)
    table['delta_meta_vs_all'] = table['meta_regressor_rf'] - table['all_images_mean']
    return table


In [ ]:
def plot_panel(ax, dataset: str):
    preds = post_preds[dataset]
    feats = post_feats[dataset]
    merged = preds[['row_id', 'iou_target', 'full__random_forest']].merge(
        feats[['row_id', 'conf_score_mean', 'tta_iou_mean']],
        on='row_id',
        how='inner',
    )

    y = merged['iou_target'].to_numpy(dtype=float)
    conf = merged['conf_score_mean'].to_numpy(dtype=float)
    tta = merged['tta_iou_mean'].to_numpy(dtype=float)
    meta = merged['full__random_forest'].to_numpy(dtype=float)

    coverages = np.linspace(0.05, 1.0, 80)
    p5, p95 = random_band(y, coverages)
    baseline = y.mean()

    ax.fill_between(coverages * 100, p5, p95, color='grey', alpha=0.18, label='Random baseline')
    ax.axhline(baseline, color='grey', linewidth=1.0, linestyle='--')
    ax.axhline(baseline, color='black', linewidth=1.0, linestyle=':', label=f'All images (μ={baseline:.3f})')

    ax.plot(coverages * 100, rc_curve(y, conf, coverages), color='green', linewidth=1.8, linestyle='-.', label='Detection confidence')
    ax.plot(coverages * 100, rc_curve(y, tta, coverages), color='steelblue', linewidth=2.0, label='TTA consistency')
    ax.plot(coverages * 100, rc_curve(y, meta, coverages), color='crimson', linewidth=2.4, label='Meta-regressor (RF)')

    ax.set_xlim(5, 100)
    ax.set_title(f'{DATASET_LABELS[dataset]} dataset', fontsize=12, fontweight='bold')
    ax.set_xlabel('Coverage (% of images accepted)')
    ax.set_ylabel('Mean IoU (accepted images)')
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, dataset in zip(axes, DATASETS):
    plot_panel(ax, dataset)

axes[1].legend(loc='lower right', fontsize=8.5, frameon=True)
fig.tight_layout()

risk_path = FIG_DIR / 'falco_risk_coverage_local.png'
fig.savefig(risk_path, dpi=160, bbox_inches='tight')
plt.show()
print(risk_path)


In [ ]:
risk_tables = pd.concat([risk_coverage_table(dataset) for dataset in DATASETS], ignore_index=True)
risk_tables


## Figure 3: Predicted vs True Quality

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

scatter_size_px = (2615, 1457)
scatter_dpi = 300
fig, axes = plt.subplots(1, 2, figsize=tuple(size / scatter_dpi for size in scatter_size_px),
                         dpi=scatter_dpi, sharex=True, sharey=True)
for ax, dataset in zip(axes, DATASETS):
    predictions = post_preds[dataset]
    assert predictions["row_id"].is_unique
    true_iou = predictions["iou_target"].to_numpy(np.float32)
    predicted_iou = predictions["full__random_forest"].to_numpy(np.float32)
    assert np.isfinite(true_iou).all() and np.isfinite(predicted_iou).all()
    assert ((true_iou >= 0) & (true_iou <= 1)).all()
    assert ((predicted_iou >= 0) & (predicted_iou <= 1)).all()
    r2 = r2_score(true_iou, predicted_iou)
    rho = spearmanr(true_iou, predicted_iou).statistic
    reference = require_row(post_results[dataset], "full", "random_forest")
    np.testing.assert_allclose([r2, rho], [reference["r2"], reference["spearman"]], atol=1e-6)

    ax.scatter(true_iou, predicted_iou, s=13, alpha=0.35,
               color=DATASET_COLORS[dataset], edgecolors="none")
    ax.plot([0, 1], [0, 1], linestyle="--", color="#333333", linewidth=1.2)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="True union-mask IoU",
           title=f"{DATASET_LABELS[dataset]} (n={len(predictions):,})")
    ax.set_xlabel("True union-mask IoU", fontsize=15)
    ax.set_title(f"{DATASET_LABELS[dataset]} (n={len(predictions):,})", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_aspect("equal", adjustable="box")
    ax.text(0.04, 0.96, f"$R^2$ = {r2:.3f}\nSpearman $\\rho$ = {rho:.3f}",
            transform=ax.transAxes, va="top", fontsize=14,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.9})

axes[0].set_ylabel("Predicted union-mask IoU", fontsize=15)
fig.tight_layout()
scatter_path = ROOT / "paper" / "plots" / "fig_scatter.png"
scatter_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(scatter_path, dpi=scatter_dpi, bbox_inches=None)
print(f"Saved {scatter_path}")
plt.show()
plt.close(fig)


In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(12, 5.3))
ax.set(xlim=(0, 12), ylim=(0, 5.3))
ax.axis('off')

def box(x, y, width, height, text, color):
    ax.add_patch(FancyBboxPatch((x, y), width, height,
                 boxstyle='round,pad=0.06', facecolor=color, edgecolor='#374151', linewidth=0.9))
    ax.text(x + width / 2, y + height / 2, text, ha='center', va='center', fontsize=10)

def arrow(start, end, dashed=False):
    ax.annotate('', xy=end, xytext=start, arrowprops={
        'arrowstyle': '->', 'color': '#59636e', 'lw': 1.2,
        'linestyle': '--' if dashed else '-'})

ax.text(3.4, 4.95, 'PRE-DETECTION: no segmentation pass', color='#245c83', fontsize=11, weight='bold')
ax.text(3.4, 2.95, 'POST-DETECTION: detector outputs required', color='#28654d', fontsize=11, weight='bold')
box(0.15, 3.1, 1.4, 0.85, 'Raw image', '#f1f3f5')
box(2.1, 3.9, 3.2, 0.8, 'Candidate ROI / image features\nHC or frozen image embeddings', '#e5f1fa')
box(6.1, 3.9, 2.6, 0.8, 'Pre-detection\nregressor', '#e5f1fa')
box(2.1, 1.9, 1.7, 0.8, 'Assigned detector\nMask R-CNN', '#e5f3eb')
box(4.35, 1.9, 2.3, 0.8, 'Confidence / TTA /\nmulti-threshold features', '#e5f3eb')
box(7.15, 1.9, 1.55, 0.8, 'Post-detection\nregressor', '#e5f3eb')
box(9.6, 2.9, 2.0, 1.0, 'Estimated quality\n(union-mask IoU)', '#f1f3f5')
box(2.1, 0.35, 1.7, 0.65, 'Ground-truth masks', '#fff0db')
box(4.35, 0.35, 2.3, 0.65, 'Shared union-IoU target', '#fff0db')
arrow((1.55, 3.75), (2.1, 4.3))
arrow((1.55, 3.3), (2.1, 2.3))
arrow((5.3, 4.3), (6.1, 4.3))
arrow((3.8, 2.3), (4.35, 2.3))
arrow((6.65, 2.3), (7.15, 2.3))
arrow((8.7, 4.3), (9.6, 3.7))
arrow((8.7, 2.3), (9.6, 3.1))
arrow((3.8, 0.675), (4.35, 0.675), True)
arrow((2.95, 1.9), (4.8, 1.0), True)
ax.text(7.0, 0.4, 'Shared target for both regressors\nFit on outer training folds; evaluate on test fold\nDashed: target construction only', fontsize=9, color='#59636e')
fig.tight_layout()
fig.savefig(ROOT / 'paper' / 'plots' / 'fig_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
